In [ ]:
import ROOT

import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as interp
import scipy.optimize as opt
import matplotlib as mpl
import scipy.special as sf
import scipy.integrate as integ
import scipy.linalg as la
mpl.rc('xtick', direction='in', top=True)
mpl.rc('ytick', direction='in', right=True)
mpl.rc('xtick.minor', visible=True)
mpl.rc('ytick.minor', visible=True)
import pandas as pd
import os
import re
import uuid
import json
import copy



In [122]:
base_path1="Band_class_training data/TE_011/Results"
index_path1 = os.path.join(base_path1, "Te011.json")
with open(index_path1, "r") as f:
    TE011_Index = json.load(f)

In [ ]:
base_path2 = "Band_class_training data/TE_012/Results"
index_path2 = os.path.join(base_path2, "Te012.json")
with open(index_path2, "r") as f:
    TE012_Index = json.load(f)

In [124]:
def process_jason(index, angle, position):
    jsn_name='LocustEventProperties.json'
    Jsn_info = index[angle][position][jsn_name]
    filepath = Jsn_info["filepath"]

    with open(filepath, "r") as f:
        jsn = json.load(f)

    return[jsn]

In [ ]:
def process_hist(index, angle, position, hist_name,rebin=None):

    #load file
    hist_info = index[angle][position][hist_name]
    filepath = hist_info["filepath"]
    file = ROOT.TFile.Open(filepath,'READ')
    hist = file.Get(hist_name)

    #check file type
    if not hist:
        print("Missing:", hist_name)
        file.Close()
        return

    #clone it so i dont mess it up
    clone_name = f"{hist_name}_clone_{uuid.uuid4().hex}"
    hist_copy = hist.Clone(clone_name)
    hist_copy.SetDirectory(0)
    file.Close()

    if rebin is not None and rebin > 1:
        if not isinstance(rebin, int):
            raise ValueError("rebin must be an integer")
        old_level = ROOT.gErrorIgnoreLevel
        ROOT.gErrorIgnoreLevel = ROOT.kError
        hist_copy.Rebin(rebin)
        ROOT.gErrorIgnoreLevel = old_level

    nbins = hist_copy.GetNbinsX()
    counts = np.array([
            hist_copy.GetBinContent(i)
            for i in range(1, nbins + 1)
        ])
    centers=np.array([
            hist_copy.GetBinCenter(i)
            for i in range(1, nbins + 1)
        ])
    

    return(hist_copy, counts, centers)

In [126]:
def print_hist(index, angle, position, hist_name):
    hist, counts, centers=process_hist(index, angle, position, hist_name)
    del counts
    del centers
    for i in range(1, hist.GetNbinsX() + 1):
        print(hist.GetBinCenter(i), hist.GetBinContent(i))
    nbins = hist.GetNbinsX()
    print(nbins)
    hist.Delete()

In [127]:
def plot_hist(index, angle, position, hist_name, rebin=None, ax=None, max=None):

    hist, counts, centers=process_hist(index, angle, position, hist_name,rebin=rebin)
    del counts
    del centers

    nbins = hist.GetNbinsX()

    x =[]
    y =[]
    width =[]

    for i in range(1, nbins + 1):
        x.append(hist.GetBinCenter(i))
        y.append(hist.GetBinContent(i))
        width.append(hist.GetBinWidth(i))
    x = np.array(x)
    y = np.array(y)
    width = np.array(width)

    if ax is None:
        fig, ax = plt.subplots()
    else:
        fig = ax.figure

    if max is None:
        bars=ax.bar(x, y, width=width, align="center")
    else:
        lims = (x >= max[0]) & (x <= max[1])
        bars=ax.bar(x[lims], y[lims], width=width[lims], align="center")

    ax.set_xlabel("Bin center")
    ax.set_ylabel("Counts")
    ax.set_title(f"{angle}, {position}, {hist_name}")

    return fig, ax, bars

In [128]:
def max_hist(index, angle, position, hist_name, gtr=None,btwn=None):
        hist, counts, centers=process_hist(index, angle, position, hist_name)
        hist.Delete()
        maxes = []
        if btwn is not None:
            lims = (centers >= btwn[0]) & (centers <= btwn[1])
        else:
            lims = np.ones_like(counts, dtype=bool)
        counts= counts[lims]
        centers= centers[lims]  
        if gtr==None:
            mx=np.max(counts)
            i=int(np.where(counts==mx)[0][0])
            center=centers[i]
            power=counts[i]
            maxes.append([power,center])
        else:
            
            i=np.where(counts>gtr)[0]
            for a in range(len(i)):
                center=centers[int(i[a])]
                power=counts[int(i[a])]
                maxes.append([power,center])
        del counts
        del centers

        maxes=np.array(maxes)
        return(maxes)


In [129]:
def Time_density_hist(index, angle, position, rebin=None,freq_range=None):

    def get_time_from_key(key):
        nums = re.findall(r"-?\d+\.?\d*", key)
        if len(nums) == 0:
            raise ValueError(f"No time/index number found in key: {key}")
        return float(nums[0])

    psd_keys = [
        key for key in index[angle][position]
        if "PSD" in key
    ]

    psd_keys = sorted(psd_keys, key=get_time_from_key)
    times = []
    rows = []
    freq_edges = None

    for key in psd_keys:
        time = get_time_from_key(key)

        hist,counts, centers = process_hist(index, angle, position, key, rebin)
        nbins= hist.GetNbinsX()
        edges = np.array(
            [hist.GetBinLowEdge(i) for i in range(1, nbins + 1)]
            + [hist.GetBinLowEdge(nbins) + hist.GetBinWidth(nbins)])
        del centers
        hist.Delete()
        times.append(time)
        rows.append(counts)
        if freq_edges is None:
            freq_edges = edges
    times = np.array(times)
    data = np.array(rows)

    if freq_range is not None:
        freq_centers = 0.5 * (freq_edges[:-1] + freq_edges[1:])
        lims = (freq_centers >= freq_range[0]) & (freq_centers <= freq_range[1])
        idx = np.where(lims)[0]

        first = idx[0]
        last = idx[-1]

        data = data[:, first:last + 1]
        freq_edges = freq_edges[first:last + 2]
        
    data = data.T

    # Make time bin edges from time centers
    if len(times) > 1:
        time_edges = np.zeros(len(times) + 1)
        time_edges[1:-1] = 0.5 * (times[:-1] + times[1:])
        time_edges[0] = times[0] - 0.5 * (times[1] - times[0])
        time_edges[-1] = times[-1] + 0.5 * (times[-1] - times[-2])
    else:
        time_edges = np.array([times[0] - 0.5, times[0] + 0.5])

    fig, ax = plt.subplots(figsize=(8, 6))

    mesh = ax.pcolormesh(
        time_edges,
        freq_edges,
        data,
        shading="auto"
    )

    fig.colorbar(mesh, ax=ax, label="Counts")

    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency")
    ax.set_title(f"PSD density map: angle={angle}, position={position}")
    del data
    del rows
    del times


In [130]:
def find_bands(index, angle, position,hist_name,mode,spread=11):
    if spread%2 != 1:
         spread=spread+1
    jsn1=process_jason(index,angle,position)
    avgax=jsn1[0]['0']['0']['avg-axial-frequency']
    central=jsn1[0]['0']['0']['output-avg-frequency']
    samplefreq=50000

    if mode==1:
        bands=np.array([-4,-2,0,2,4])    
    elif mode==2:
        bands=np.array([-3,-1,1,3])
    sidebands=[]   
    for band in bands:
            band_loc=central+(band*avgax)
            band_range=[band_loc-(spread*samplefreq),band_loc+(spread*samplefreq)]
            sideband=max_hist(index, angle, position,hist_name,btwn=band_range)
            sidebands.append(sideband[0])
    sidebands=np.array(sidebands)       

    return(sidebands)



In [ ]:
def sideband_csv(index,name,mode):

    index_copy=copy.deepcopy(index)
    for key1 in index:
        for key2 in index[key1]:
            for key3 in index[key1][key2]:
                if "PSD" not in key3 and "json" not in key3:
                    del index_copy[key1][key2][key3]
                    continue
                if "json" not in key3:
                    sidebands=find_bands(index,key1,key2,key3,mode)
                    sidebands=sidebands.tolist()
                    index_copy[key1][key2][key3]={
                        "side_bands_powercenter":sidebands
                    }
    return(index_copy)
# this can take up to 40 minutes there is a better version in sidebands_loader

In [143]:
def save_index(index,position):
    with open(position, "w") as f:
        json.dump(index, f, indent=4)